# 3D Surface Explorer
### Orbital rotation of `f(x, y)` surfaces

---

## What is a surface plot?

A surface plot shows a function of **two variables** `f(x, y)` as a 3D shape:
- The `x` and `y` axes are the input plane (the floor of the 3D space)
- The **height** `z = f(x, y)` at each point is the output
- **Color** also encodes height via the colorbar — so you can read values from any viewing angle

### Reading the surface

| Feature | What it means |
|---|---|
| **Peak** (local maximum) | `∂f/∂x = 0` and `∂f/∂y = 0`, surface curves downward in all directions |
| **Valley** (local minimum) | Same conditions, surface curves upward |
| **Saddle point** | Curves up in one direction, down in another — looks like a mountain pass |
| **Flat ridge** | Function constant along one axis, varying along the other |
| **Oscillating** | Trig functions like `sin(x)*cos(y)` — periodic in both directions |
| **Decaying amplitude** | Multiplying by `exp(-(x²+y²))` shrinks the surface toward zero far from origin |

### How orbital rotation works internally

Each frame **removes and re-adds the surface mesh** so matplotlib recomputes polygon depth-sorting for the current camera angle. Without this, back-facing polygons bleed through the front as the surface rotates. The camera orbits by incrementing the azimuth angle each frame; with the default settings, one render = one full 360° orbit.

---

## Example equations

| Equation | What you see | Best x/y range |
|---|---|---|
| `sin(sqrt(x**2 + y**2))` | Circular ripples spreading from origin | `-6` to `6` |
| `exp(-0.1*(x**2 + y**2)) * cos(x + y)` | Gaussian-modulated diagonal wave | `-5` to `5` |
| `sin(x) * cos(y)` | Saddle-wave egg-carton grid | `-4` to `4` |
| `x * exp(-x**2 - y**2)` | Ricker wavelet — one positive/negative ridge | `-3` to `3` |
| `(1 - 2*(x**2+y**2)) * exp(-(x**2+y**2))` | Mexican hat / Laplacian of Gaussian | `-3` to `3` |
| `3*(1-x)**2*exp(-x**2-(y+1)**2) - 10*(x/5-x**3-y**5)*exp(-x**2-y**2) - (1/3)*exp(-(x+1)**2-y**2)` | MATLAB peaks — three bumps and a valley | `-3` to `3` |
| `x**2 / 4 - y**2 / 4` | Hyperbolic saddle | `-4` to `4` |
| `sin(x) * y` | Twisted ribbon | `-4` to `4` |
| `cos(x**2 + y**2)` | Concentric rings (harder edges than ripple) | `-3` to `3` |
| `sin(x**2 + y**2) / (x**2 + y**2 + 0.1)` | Damped 2D sinc | `-5` to `5` |

---

## Parameters guide

| Control | Effect |
|---|---|
| **Resolution** | Grid density — higher = smoother surface, slower render. 40 is fast, 70 is detailed. |
| **FPS** | Frames per second — 20 is smooth for jshtml inline playback. |
| **Duration** | Total animation length. With default spin speed, `8 s = one full orbit`. |
| **Spin speed** | Degrees rotated per frame. `2.25°/frame × 20fps × 8s = 360°`. |
| **Elevation** | Camera height angle above horizontal. 28° = comfortable isometric view; 60°+ = top-down. |
| **Color map** | `viridis`/`plasma` on dark backgrounds; `coolwarm`/`RdBu` for signed functions (zero = white). |

---

In [ ]:
import matplotlib
matplotlib.rcParams['animation.embed_limit'] = 100

import ipywidgets as w
from IPython.display import display, HTML, clear_output
import mathplt.animations
from mathplt.animations.graph3d import Graph3DAnimator
from mathplt.core.animator import AnimationConfig
from mathplt.jupyter.widgets import AnimationWidget
from mathplt.core.equation_parser import EquationParser

display(HTML("""
<style>
  .mp-panel  { background:#161b27; border:1px solid #2d3748; border-radius:10px;
               padding:18px 20px; margin-bottom:14px; }
  .mp-section{ color:#718096; font-size:11px; font-weight:700; letter-spacing:.08em;
               text-transform:uppercase; margin:0 0 8px; }
  .mp-hint   { color:#4a5568; font-size:12px; margin-top:6px; }
  .mp-tag    { display:inline-block; background:#2d3748; color:#a0aec0; border-radius:4px;
               padding:2px 7px; font-size:11px; margin:2px; cursor:default; }
</style>
<div class='mp-panel'>
  <div style='font-size:22px;font-weight:700;color:#e2e8f0;margin-bottom:4px'>3D Surface Explorer</div>
  <div style='color:#718096;font-size:13px'>Orbital rotation of <code>f(x, y)</code> surfaces with correct depth-sorting per frame.</div>
</div>
"""))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  Preset equations with descriptions shown on hover
# ─────────────────────────────────────────────────────────────────────────────
PRESETS = [
    ("Ripple",         "sin(sqrt(x**2 + y**2))",          (-6, 6),  (-6, 6)),
    ("Gaussian wave",  "exp(-0.1*(x**2 + y**2)) * cos(x + y)", (-5, 5), (-5, 5)),
    ("Saddle wave",    "sin(x) * cos(y)",                 (-4, 4),  (-4, 4)),
    ("Ricker wavelet", "x * exp(-x**2 - y**2)",           (-3, 3),  (-3, 3)),
    ("Mexican hat",    "(1 - 2*(x**2+y**2)) * exp(-(x**2+y**2))", (-3, 3), (-3, 3)),
    ("Peaks",          "3*(1-x)**2*exp(-x**2-(y+1)**2) - 10*(x/5-x**3-y**5)*exp(-x**2-y**2) - (1/3)*exp(-(x+1)**2-y**2)", (-3, 3), (-3, 3)),
    ("Hyperbolic",     "x**2 / 4 - y**2 / 4",            (-4, 4),  (-4, 4)),
    ("Twisted ribbon", "sin(x) * y",                      (-4, 4),  (-4, 4)),
    ("Sinc 2D",        "sin(x**2 + y**2) / (x**2 + y**2 + 0.1)", (-5, 5), (-5, 5)),
]

CMAPS = [
    "viridis", "plasma", "inferno", "magma", "cividis",
    "coolwarm", "RdBu", "turbo", "twilight", "hsv",
]

# ─────────────────────────────────────────────────────────────────────────────
#  Widgets
# ─────────────────────────────────────────────────────────────────────────────
style = {"description_width": "130px"}
lh = w.Layout(width="48%")
lf = w.Layout(width="100%")

eq_input = w.Text(
    value=PRESETS[0][1],
    placeholder="e.g. sin(sqrt(x**2 + y**2))",
    description="f(x, y) =",
    style=style,
    layout=w.Layout(width="600px"),
)
eq_status = w.HTML(value='<span style="color:#68d391">✓ Valid</span>')

# Preset buttons — clicking also updates domain sliders
def _make_preset_btn(name, eq, xr, yr):
    btn = w.Button(description=name, layout=w.Layout(width="auto", margin="2px"))
    def _on_click(_):
        eq_input.value = eq
        x_min.value, x_max.value = xr
        y_min.value, y_max.value = yr
    btn.on_click(_on_click)
    return btn

preset_buttons = [_make_preset_btn(*p) for p in PRESETS]

x_min   = w.FloatSlider(value=-6, min=-15, max=-0.5, step=0.5, description="x min",       style=style, layout=lh)
x_max   = w.FloatSlider(value=6,  min=0.5, max=15,   step=0.5, description="x max",       style=style, layout=lh)
y_min   = w.FloatSlider(value=-6, min=-15, max=-0.5, step=0.5, description="y min",       style=style, layout=lh)
y_max   = w.FloatSlider(value=6,  min=0.5, max=15,   step=0.5, description="y max",       style=style, layout=lh)
res_sl  = w.IntSlider(  value=50, min=20,  max=100,  step=5,   description="Resolution",  style=style, layout=lh)
fps_sl  = w.IntSlider(  value=20, min=8,   max=30,   step=2,   description="FPS",         style=style, layout=lh)
dur_sl  = w.FloatSlider(value=8,  min=3,   max=24,   step=1,   description="Duration (s)",style=style, layout=lh)
spd_sl  = w.FloatSlider(value=2.25,min=0.5,max=6,    step=0.25,description="Spin speed°", style=style, layout=lh)
elev_sl = w.FloatSlider(value=28, min=5,   max=85,   step=1,   description="Elevation°",  style=style, layout=lh)
cmap_dd = w.Dropdown(   options=CMAPS, value="viridis",         description="Color map",   style=style, layout=lh)
dark_cb = w.Checkbox(   value=True,    description="Dark mode",  style=style, layout=lh)

# Orbit-info label — shows total rotation given current settings
orbit_label = w.HTML()

def _update_orbit_label(*_):
    total = round(fps_sl.value * dur_sl.value * spd_sl.value, 0)
    orbits = total / 360
    orbit_label.value = (
        f'<span style="color:#718096;font-size:12px">'
        f'Total rotation: <b style="color:#a0aec0">{total:.0f}°</b> '
        f'= <b style="color:#a0aec0">{orbits:.2f}</b> full orbits'
        f'</span>'
    )

for sl in (fps_sl, dur_sl, spd_sl):
    sl.observe(_update_orbit_label, names="value")
_update_orbit_label()

# Live equation validation
_parser = EquationParser()

render_btn = w.Button(
    description=" Render Animation",
    button_style="success",
    icon="play",
    layout=w.Layout(width="210px", height="40px"),
)
output = w.Output()

def _validate(change=None):
    err = _parser.validate(eq_input.value, variables={"x", "y"})
    if err:
        eq_status.value = f'<span style="color:#fc8181">✗ {err}</span>'
        render_btn.disabled = True
    else:
        eq_status.value = '<span style="color:#68d391">✓ Valid</span>'
        render_btn.disabled = False

eq_input.observe(_validate, names="value")

# ─────────────────────────────────────────────────────────────────────────────
#  Render callback
# ─────────────────────────────────────────────────────────────────────────────
def _render(_btn):
    import matplotlib.pyplot as plt
    with output:
        clear_output(wait=True)
        display(HTML('<span style="color:#a0aec0">Rendering — please wait…</span>'))

        plt.close("all")
        config = AnimationConfig(
            fps=fps_sl.value,
            duration_seconds=dur_sl.value,
            figsize=(11, 7),
            dpi=90,
            dark_mode=dark_cb.value,
        )
        animator = Graph3DAnimator(
            config,
            equation=eq_input.value,
            x_range=(x_min.value, x_max.value),
            y_range=(y_min.value, y_max.value),
            resolution=res_sl.value,
            cmap=cmap_dd.value,
            azim_per_frame=spd_sl.value,
            elev=elev_sl.value,
        )
        clear_output(wait=True)
        AnimationWidget(animator).display()

render_btn.on_click(_render)

# ─────────────────────────────────────────────────────────────────────────────
#  Layout
# ─────────────────────────────────────────────────────────────────────────────
def _section(label):
    return w.HTML(f"<div class='mp-section'>{label}</div>")

def _hint(text):
    return w.HTML(f"<div class='mp-hint'>{text}</div>")

def _divider():
    return w.HTML("<hr style='border:none;border-top:1px solid #2d3748;margin:10px 0'>")

display(w.VBox([
    # ── Equation ──────────────────────────────────────────────────────────
    _section("Equation   f(x, y)"),
    w.HBox([eq_input, eq_status], layout=w.Layout(align_items="center")),
    _hint("Use <code>x</code>, <code>y</code>, and any of: "
          "<code>sin cos tan exp log sqrt abs pi</code>. "
          "Operator precedence follows Python — use <code>**</code> for powers."),

    _divider(),

    # ── Presets ───────────────────────────────────────────────────────────
    _section("Presets  (click to load equation + domain)"),
    w.HBox(preset_buttons, layout=w.Layout(flex_flow="row wrap")),

    _divider(),

    # ── Domain ────────────────────────────────────────────────────────────
    _section("Domain"),
    w.HBox([x_min, x_max]),
    w.HBox([y_min, y_max]),

    _divider(),

    # ── Render settings ───────────────────────────────────────────────────
    _section("Render"),
    w.HBox([res_sl, fps_sl]),
    _hint("Resolution 40–50 renders fast; use 70+ for a final high-quality render."),
    w.HBox([dur_sl, spd_sl]),
    orbit_label,

    _divider(),

    # ── View settings ─────────────────────────────────────────────────────
    _section("View"),
    w.HBox([elev_sl, cmap_dd]),
    _hint("<b>Elevation 28°</b> = isometric-style view &nbsp;|&nbsp; "
          "<b>viridis/plasma</b> on dark &nbsp;|&nbsp; "
          "<b>coolwarm/RdBu</b> best for signed surfaces &nbsp;|&nbsp; "
          "<b>turbo</b> for high-contrast."),
    dark_cb,

    w.HTML("<div style='height:10px'></div>"),
    render_btn,
    output,
], layout=w.Layout(padding="6px")))